# A armadilha e o resto do cardapioEquivale aos scripts `06_armadilha.sql` e `07_modelos_prontos.sql`.---**PDM 2026.2 — aula de 04/09/2026 · notebook 3 de 3**Este notebook e o mesmo conteudo dos scripts `.sql` da aula, so que rodando de dentrodo Python. Serve para estudar sozinho, para repetir a aula no seu ritmo e como pontepara a proxima aula, em que o dado sai do BigQuery e vira DataFrame.Voce pode rodar tudo no [Google Colab](https://colab.research.google.com) sem instalar nada.Se rodar na sua maquina, precisa de `pip install google-cloud-bigquery pandas db-dtypes`e de um `gcloud auth application-default login` antes.Uma coisa de cada vez: **nao rode tudo de uma vez**. Rode uma celula, leia o resultado,so entao passe para a proxima. O valor da aula esta em olhar para o numero que aparece.

In [ ]:
# Rode este bloco UMA VEZ, antes de qualquer outro.# Ele autentica voce no Google Cloud e cria os atalhos q(...) e run(...).PROJETO = "SEU_PROJETO"      # <<< troque pelo id do SEU projeto no GCPDATASET = "anuncios"LOCAL   = "us-central1"      # a regiao do dataset. Precisa bater, senao o BigQuery recusa.try:    from google.colab import auth    auth.authenticate_user()          # Colab: abre a janela de login do Googleexcept ImportError:    pass                              # local: rode antes `gcloud auth application-default login`import pandas as pdfrom google.cloud import bigqueryclient = bigquery.Client(project=PROJETO, location=LOCAL)def q(sql: str) -> pd.DataFrame:    """Roda o SQL e devolve um DataFrame. Toda ocorrencia de SEU_PROJETO vira o seu projeto."""    return client.query(sql.replace("SEU_PROJETO", PROJETO)).to_dataframe()def run(sql: str) -> None:    """Para CREATE TABLE / CREATE MODEL: executa, nao devolve tabela."""    client.query(sql.replace("SEU_PROJETO", PROJETO)).result()    print("ok")pd.set_option("display.float_format", lambda v: f"{v:,.2f}")print("conectado em", PROJETO, "|", LOCAL)

## 1. Vazamento de alvoNo notebook 2 voce extraiu seis features do `titulo` com `REGEXP` e o modelo melhorou.Agora voce vai extrair **mais uma** feature do mesmo `titulo`, com a **mesma funcao**,e o modelo vai melhorar muito mais.E vai ser uma fraude. Comece olhando o dado.

In [ ]:
q(r"""SELECT  titulo,  ROUND(preco) AS preco,  REGEXP_EXTRACT(titulo, r'R\$\s*([\d\.]+(?:,\d+)?)') AS valor_no_tituloFROM `SEU_PROJETO.anuncios.anuncios_gold`WHERE REGEXP_CONTAINS(titulo, r'R\$')LIMIT 15""")

**Como ler.** Compare `valor_no_titulo` com `preco`, linha a linha. Em boa parte doscasos e o mesmo numero: o anunciante escreveu o preco dentro do titulo do anuncio.Do ponto de vista do SQL, extrair isso e identico a extrair "piscina". Mesma coluna,mesma funcao, mesma sintaxe. Nada no codigo avisa que voce cruzou uma linha.

In [ ]:
run(r"""CREATE OR REPLACE TABLE `SEU_PROJETO.anuncios.gold_com_titulo` ASSELECT  *,  CASE    WHEN REGEXP_EXTRACT(titulo, r'R\$\s*([\d\.]+(?:,\d+)?)') LIKE '%,%'      THEN SAFE_CAST(REPLACE(REPLACE(             REGEXP_EXTRACT(titulo, r'R\$\s*([\d\.]+(?:,\d+)?)'),             '.', ''), ',', '.') AS FLOAT64)    ELSE SAFE_CAST(           REGEXP_EXTRACT(titulo, r'R\$\s*([\d\.]+(?:,\d+)?)') AS FLOAT64)  END AS preco_no_tituloFROM `SEU_PROJETO.anuncios.anuncios_gold`""")

In [ ]:
q("""SELECT  COUNT(*)                                             AS total,  COUNTIF(preco_no_titulo IS NOT NULL)                 AS com_preco_no_titulo,  COUNTIF(ABS(preco_no_titulo - preco) < preco * 0.02) AS valor_bate_com_precoFROM `SEU_PROJETO.anuncios.gold_com_titulo`""")

**Como ler.** `valor_bate_com_preco` conta em quantas linhas o numero do titulo e opreco real sao praticamente iguais. Nessas linhas, a "feature" **e** o alvo, escritocom outras palavras. Guarde essa contagem: ela vai explicar o resultado do modelo.

In [ ]:
run("""CREATE OR REPLACE MODEL `SEU_PROJETO.anuncios.modelo_preco_v2`OPTIONS (  model_type            = 'LINEAR_REG',  input_label_cols      = ['preco'],  data_split_method     = 'AUTO_SPLIT',  enable_global_explain = TRUE) ASSELECT  preco,  area_util, quartos, banheiros, suites, garagem, condominio, iptu,  bairro, eh_comercial,  preco_no_tituloFROM `SEU_PROJETO.anuncios.gold_com_titulo`""")

In [ ]:
q("""SELECT 'v1_sem_titulo' AS modelo, mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco`)UNION ALLSELECT 'v2_com_titulo', mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_v2`)""")

**Como ler.** O v2 e melhor que o v1 em qualquer metrica que voce olhe. Se o criteriode aprovacao for "o R2 subiu", ele passa. E ele nao serve para nada.O motivo: no momento em que voce quer usar o modelo — um imovel novo, ainda sem precodefinido — nao existe preco escrito no titulo. A feature so existe depois que aresposta ja e conhecida. Voce treinou um modelo que so funciona quando nao precisa dele.

In [ ]:
q("""SELECT *FROM ML.GLOBAL_EXPLAIN(MODEL `SEU_PROJETO.anuncios.modelo_preco_v2`)ORDER BY attribution DESC""")

**Como ler.** `preco_no_titulo` domina o ranking com folga sobre todo o resto. Umafeature que sozinha vale mais que a soma de area, bairro e quartos e sempre suspeita.Esse desequilibrio no `GLOBAL_EXPLAIN` e o alarme mais confiavel de vazamento quevoce tem em maos.Confirme separando os dois grupos.

In [ ]:
q("""SELECT  preco_no_titulo IS NOT NULL              AS titulo_entregava_o_preco,  COUNT(*)                                 AS anuncios,  ROUND(AVG(ABS(predicted_preco - preco))) AS erro_medio_absolutoFROM ML.PREDICT(  MODEL `SEU_PROJETO.anuncios.modelo_preco_v2`,  (SELECT * FROM `SEU_PROJETO.anuncios.gold_com_titulo`))GROUP BY 1""")

**Como ler.** Nas linhas em que o titulo trazia o preco, o erro despenca; nas outras,volta ao patamar do v1. O modelo nao aprendeu nada sobre imoveis. Ele aprendeu acopiar um numero.### A pergunta que separa feature de vazamentoNao e uma questao de tecnica, e uma questao de tempo:> **Essa informacao existiria no momento em que eu preciso da previsao?**- `eh_apartamento`, `eh_terreno`, `tem_piscina` — sim. Um anuncio novo ja diz o tipo do imovel.- `preco_no_titulo` — nao. Se o preco ja esta escrito, nao ha o que prever.Mesma coluna de origem, mesma funcao do BigQuery, resultados moralmente opostos.Nenhum validador automatico separa os dois casos. Voce separa.---## 2. Alavanca 3 — trocar o algoritmo (v4)Ate aqui foi tudo `LINEAR_REG`, que so consegue tracar retas. Preco de imovel nao euma reta: dobrar a area nao dobra o preco, e o efeito do bairro muda conforme o tipo.Arvore com boosting nao tem esse limite. E, no BQML, trocar de familia de algoritmoe trocar **uma string**.

In [ ]:
run("""CREATE OR REPLACE MODEL `SEU_PROJETO.anuncios.modelo_preco_arvore`OPTIONS (  model_type            = 'BOOSTED_TREE_REGRESSOR',  input_label_cols      = ['preco'],  data_split_method     = 'AUTO_SPLIT',  enable_global_explain = TRUE) ASSELECT  preco,  bairro, area_util, quartos, banheiros, suites, garagem, condominio, iptu,  eh_comercial,  tem_piscina, eh_alto_padrao, tem_mobilia,  eh_terreno, eh_apartamento, eh_casaFROM `SEU_PROJETO.anuncios.gold_texto`""")

> Este treino demora mais que os anteriores. E esperado: a arvore ajusta centenas de> modelos pequenos em sequencia, em vez de resolver uma equacao de uma vez.**Como ler o OPTIONS.** Mesmo alvo, mesmas colunas, mesma tabela do v3. Uma unicapalavra diferente. Guarde isso quando alguem disser que trocar de modelo e um projeto novo.

In [ ]:
placar = q("""SELECT 'v0_dado_como_estava'  AS modelo, mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_imoveis`)UNION ALLSELECT 'v1_gold_limpa',        mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco`)UNION ALLSELECT 'v3_gold_mais_texto',   mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_v3`)UNION ALLSELECT 'v4_arvore_mesmo_dado', mean_absolute_error, r2_scoreFROM ML.EVALUATE(MODEL `SEU_PROJETO.anuncios.modelo_preco_arvore`)ORDER BY r2_score""")placar

**Como ler.** Tres alavancas, tres saltos, em ordem decrescente de tamanho:1. **limpar o dado** (v0 -> v1) — o maior salto de todos;2. **criar feature** (v1 -> v3) — o segundo;3. **trocar o algoritmo** (v3 -> v4) — o terceiro.A que menos parece machine learning foi a que mais pagou. Essa e a tese da aula.

In [ ]:
import matplotlib.pyplot as pltordem = ["v0_dado_como_estava", "v1_gold_limpa", "v3_gold_mais_texto", "v4_arvore_mesmo_dado"]p = placar.set_index("modelo").loc[ordem].reset_index()fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))ax1.plot(p["modelo"], p["mean_absolute_error"] / 1e6, marker="o", color="#c0392b")ax1.set_title("MAE em R$ milhoes — quanto menor, melhor")ax1.tick_params(axis="x", rotation=25)ax2.plot(p["modelo"], p["r2_score"], marker="o", color="#27ae60")ax2.axhline(0, color="#333", linewidth=1, linestyle="--")ax2.set_title("R2 — a linha tracejada e chutar a media")ax2.tick_params(axis="x", rotation=25)plt.tight_layout()plt.show()

Compare o que a reta e a arvore acham importante. Sao os mesmos dados.

In [ ]:
q("""SELECT feature, ROUND(attribution, 0) AS atribuicaoFROM ML.GLOBAL_EXPLAIN(MODEL `SEU_PROJETO.anuncios.modelo_preco_v3`)ORDER BY attribution DESC""")

In [ ]:
q("""SELECT feature, ROUND(attribution, 0) AS atribuicaoFROM ML.GLOBAL_EXPLAIN(MODEL `SEU_PROJETO.anuncios.modelo_preco_arvore`)ORDER BY attribution DESC""")

**Como ler.** Os dois rankings nao coincidem. Uma coluna que a reta consideravarelevante pode aparecer com atribuicao zero na arvore, e vice-versa. "Importancia defeature" nao e uma propriedade do dado: e uma propriedade **do modelo** sobre o dado.Quem troca de algoritmo troca tambem de explicacao.---## 3. Aprendizado sem gabarito — KMEANSTodos os modelos ate aqui tinham `input_label_cols`: alguem sabia a resposta certa.Tire essa linha e o problema muda de natureza.

In [ ]:
run("""CREATE OR REPLACE MODEL `SEU_PROJETO.anuncios.segmentos_imoveis`OPTIONS (  model_type           = 'KMEANS',  num_clusters         = 4,  standardize_features = TRUE) ASSELECT  preco,  area_util,  quartos,  banheiros,  garagemFROM `SEU_PROJETO.anuncios.gold_texto`""")

**Como ler o OPTIONS.** Repare no que **nao** esta ali: nao ha `input_label_cols`.Ninguem disse ao modelo o que e certo. Ele so agrupa o que e parecido.- `num_clusters = 4` — quantos grupos voce quer. E um chute seu, nao um resultado.- `standardize_features = TRUE` — obrigatorio aqui. Sem isso, `preco` (na casa dos  milhoes) domina `quartos` (na casa das unidades) e os grupos saem so por faixa de preco.

In [ ]:
q("""SELECT  CENTROID_ID                                                 AS segmento,  COUNT(*)                                                    AS imoveis,  CAST(APPROX_QUANTILES(preco, 100)[OFFSET(50)] AS INT64)     AS preco_mediano,  CAST(APPROX_QUANTILES(area_util, 100)[OFFSET(50)] AS INT64) AS area_mediana,  ROUND(AVG(quartos), 1)                                      AS quartos_medio,  ROUND(AVG(garagem), 1)                                      AS vagas_media,  ROUND(100 * AVG(CAST(eh_comercial AS INT64)), 1)            AS pct_comercialFROM ML.PREDICT(       MODEL `SEU_PROJETO.anuncios.segmentos_imoveis`,       TABLE `SEU_PROJETO.anuncios.gold_texto`)GROUP BY segmentoORDER BY preco_mediano""")

**Como ler.** O modelo devolve numeros de grupo, nao nomes. O trabalho de dar nome eseu: olhe `area_mediana`, `quartos_medio` e `pct_comercial` juntos e cada segmentoganha uma descricao em portugues.O ponto: o KMEANS **reconstruiu** uma distincao que o schema tinha perdido. Ninguemdisse a ele o que e apartamento, casa ou galpao. Ele separou assim mesmo, porque adiferenca estava nos numeros o tempo todo.---## 4. O resto do cardapioTudo que voce viu foi uma string diferente em `model_type`. O BQML tem varias outras:| `model_type` | para que serve ||---|---|| `LINEAR_REG` | prever numero, relacao aproximadamente linear || `LOGISTIC_REG` | classificar em duas ou mais categorias || `BOOSTED_TREE_REGRESSOR` / `_CLASSIFIER` | tabelas com relacoes nao lineares || `RANDOM_FOREST_REGRESSOR` / `_CLASSIFIER` | alternativa robusta as arvores com boosting || `DNN_REGRESSOR` / `_CLASSIFIER` | rede neural sobre dado tabular || `KMEANS` | agrupar sem gabarito || `PCA` | reduzir dimensionalidade || `MATRIX_FACTORIZATION` | sistemas de recomendacao || `ARIMA_PLUS` | series temporais e previsao de demanda || `AUTOML_REGRESSOR` / `_CLASSIFIER` | o Google escolhe e ajusta o modelo por voce |Trocar de familia e um `OPTIONS`. Decidir **qual pergunta vale a pena responder**continua sendo problema seu.---## 5. DesafiosSem resposta pronta. Sao boas sementes de trabalho.1. **Rode o KMEANS com 3 e com 6 grupos.** Os segmentos ficam mais uteis ou mais   arbitrarios? Qual numero voce defenderia, e com que argumento?2. **Treine uma `BOOSTED_TREE_CLASSIFIER` para `eh_comercial`** e compare com a   regressao logistica do notebook 2. O ganho compensa o tempo de treino?3. **Crie uma flag de texto nova** (varanda, gourmet, condominio fechado, portaria 24h)   e meça o efeito dela no R2. Antes de rodar, escreva sua aposta em um papel.4. **Retreine o v4 sem a coluna `bairro`.** Quanto do desempenho vinha so de localizacao?5. **Use o v4 para prever o preco de um imovel inventado** e depois olhe o   `GLOBAL_EXPLAIN`. A previsao faz sentido para quem conhece Goiania?---## O que fica- Metrica melhor nao e modelo melhor. Vazamento sempre parece um bom resultado.- Uma feature que domina o `GLOBAL_EXPLAIN` sozinha e suspeita ate prova em contrario.- A pergunta que separa feature de vazamento e sobre **tempo**, nao sobre codigo.- Trocar de familia de algoritmo e trocar uma string.- Importancia de feature e propriedade do modelo, nao do dado.- Sem `input_label_cols`, o problema deixa de ter resposta certa — e voce vira o  responsavel por nomear o que saiu.